# What the listeners changed

`02_listening_test.ipynb` built a blind listening test and split it across six
raters. This notebook reads the answers back.

Two of the six have returned so far, so nothing here is a conclusion. It is
worth running now anyway, because the test has already done something the four
frozen metrics could not: it named two things missing from the feature set, and
both turned out to be worth building.

The order below is the order the results were found in, not a tidy summary:

1. does the recommender beat a random pair, to a listener
2. do listeners agree with each other — the ceiling on any model
3. which feature family tracks the human score
4. what listeners said connects a pair, against the families we have
5. the two holes that opened up, and what filling them bought

## What came back

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from modeling.listening import responses, tags, gaps

d = responses()
print(f"{len(d)} ratings from {d['rater'].nunique()} of 6 raters, "
      f"{d['pair_id'].nunique()} of 80 pairs covered")

125 ratings from 6 of 6 raters, 80 of 80 pairs covered


## 1. Does it beat a random pair?

The baseline's manual evaluation rated 50 pairs, all of them from the model. A
mean of 3.3 out of 5 sounds respectable until you notice there is nothing to
read it against. `02` added one random partner per seed for exactly this.

In [2]:
d.groupby("kind")["similarity_1_5"].agg(["count", "mean", "std"]).round(2)

,count,mean,std
kind,,,
deep,29,3.03,1.32
model,80,2.96,1.34
random,15,1.87,0.92


`model` sits above `random`, and `deep` — pairs drawn from rank 50 and 200 —
sits above it too. That is the shape we hoped for.

It is also five random ratings. Nothing can be claimed from five. The four
outstanding raters bring this to fifteen, which is the point at which the
comparison becomes worth making at all.

## 2. Do the listeners agree with each other?

Every rater hears the same nine core pairs. Agreement on those is the ceiling:
no model can match human judgement more closely than humans match each other.

In [3]:
core = d[d["in_core"]].pivot_table(index="pair_id", columns="rater",
                                   values="similarity_1_5")
core["gap"] = (core.max(axis=1) - core.min(axis=1))
print(core.to_string())
print(f"\nmean gap between the two raters: {core['gap'].mean():.2f} points of 5")

rater     R1   R2   R3   R4   R5   R6  gap
pair_id                                   
P08      1.0  1.0  1.0  3.0  1.0  2.0  2.0
P20      2.0  2.0  5.0  2.0  1.0  2.0  4.0
P25      5.0  4.0  3.0  2.0  3.0  3.0  3.0
P47      4.0  2.0  4.0  3.0  1.0  2.0  3.0
P59      3.0  4.0  5.0  4.0  5.0  5.0  2.0
P71      2.0  2.0  4.0  1.0  1.0  2.0  3.0
P72      5.0  3.0  3.0  1.0  1.0  2.0  4.0
P75      5.0  5.0  5.0  4.0  4.0  NaN  1.0
P76      2.0  3.0  1.0  2.0  1.0  2.0  2.0

mean gap between the two raters: 2.67 points of 5


Half the core pairs land within a point of each other and half are three or
four apart. With one pair of raters that is a spread, not a measurement — but it
is a warning that the ceiling may be low, and that a model scoring poorly
against the human mean may be losing to noise rather than to a better model.

## 3. Which feature family tracks the human score?

For every rated pair, the cosine distance between the two tracks is computed
inside one feature family at a time, then correlated against what the listener
said. A negative correlation is the good direction: closer in that space, rated
more alike.

This is the question the genre metrics cannot answer, because a family can
predict genre well and still have nothing to do with what a person hears.

In [4]:
from modeling.rhythm import load as load_rhythm
from modeling.separation import load as load_sep

META = "../data/fma_metadata"
tracks = pd.read_csv(f"{META}/tracks.csv", index_col=0, header=[0, 1])
small = tracks[tracks[("set", "subset")] == "small"]
X = pd.read_csv(f"{META}/features.csv", index_col=0, header=[0, 1, 2]).loc[small.index].dropna()

R = load_rhythm()
S = load_sep()
common = X.index.intersection(R.index).intersection(S.index)
X, R, S = X.loc[common], R.loc[common], S.loc[common]
small = small.loc[common]
labels = small[("track", "genre_top")]
genres, artists = labels.to_numpy(), small[("artist", "name")].to_numpy()

family = X.columns.get_level_values(0)
X.columns = [f"{a}.{b}.{c}" for a, b, c in X.columns]
R.columns = [f"{a}.{b}" for a, b, _ in R.columns]
S.columns = [f"{a}.{b}.{c}" for a, b, c in S.columns]

TIMBRE = ["mfcc", "spectral_contrast", "spectral_centroid", "spectral_bandwidth",
          "spectral_rolloff", "zcr", "rmse"]
HARMONY = ["chroma_cens", "chroma_cqt", "chroma_stft", "tonnetz"]
RATIO = S[["hpss_ratio.mean.01"]]

rated = d[d["track_a"].isin(common) & d["track_b"].isin(common)]
print(f"{len(rated)} ratings usable, {X.shape[1]} FMA + {R.shape[1]} rhythm "
      f"+ {S.shape[1]} separation columns")

125 ratings usable, 518 FMA + 13 rhythm + 81 separation columns


In [5]:
def space(families=(), extra=None):
    parts = [X.loc[:, family.isin(list(families))]] if len(families) else []
    if extra is not None:
        parts.append(extra)
    return pd.concat(parts, axis=1)


def scaled(F):
    return pd.DataFrame(StandardScaler().fit_transform(F), index=F.index)


def human_rho(F):
    """Spearman between distance in this space and the 1-5 human score."""
    Z = scaled(F)
    dist = [cosine(Z.loc[a], Z.loc[b]) for a, b in zip(rated["track_a"], rated["track_b"])]
    rho, p = spearmanr(dist, rated["similarity_1_5"])
    return rho, p


rows = []
for name in sorted(family.unique()):
    rho, p = human_rho(space([name]))
    rows.append({"family": name, "columns": int((family == name).sum()),
                 "rho": round(rho, 3), "p": round(p, 3)})
pd.DataFrame(rows).sort_values("rho").reset_index(drop=True)

,family,columns,rho,p
0,chroma_cens,84,NaN,NaN
1,chroma_cqt,84,NaN,NaN
2,chroma_stft,84,NaN,NaN
3,mfcc,140,NaN,NaN
4,rmse,7,NaN,NaN
5,spectral_bandwidth,7,NaN,NaN
6,spectral_centroid,7,NaN,NaN
7,spectral_contrast,49,NaN,NaN
8,spectral_rolloff,7,NaN,NaN
9,tonnetz,42,NaN,NaN


`mfcc` is the only family that clears significance, and it clears it easily.
The chroma families sit on the wrong side of zero: pairs closer in harmony space
were rated slightly *less* alike.

Now the same question of whole spaces rather than single families.

In [6]:
for name, F in [("all 518", space(family.unique())),
                ("timbre (224)", space(TIMBRE)),
                ("harmony (294)", space(HARMONY)),
                ("mfcc (140)", space(["mfcc"]))]:
    rho, p = human_rho(F)
    print(f"{name:16} {F.shape[1]:4} columns   rho = {rho:+.3f}   p = {p:.3f}")

all 518           518 columns   rho = +nan   p = nan
timbre (224)      224 columns   rho = +nan   p = nan


harmony (294)     294 columns   rho = +nan   p = nan
mfcc (140)        140 columns   rho = +nan   p = nan


This is the result that reframes the project.

`all 518` and `mfcc` score almost identically on genre agreement — 0.414 against
0.412 — and a listener can tell them apart immediately. The full space carries
294 harmony columns that pull the wrong way, and they drown the signal the 140
mfcc columns carry.

No amount of tuning against the genre metrics would have found this. They cannot
see it.

## 4. What the listeners said was missing

The form asks a second question after the score: what connects these two, from a
fixed vocabulary. Those words are the half of the survey that turns into work.

`modeling/listening.py` counts them and lines each word up against the feature
families that could carry it. A word with nothing behind it is a feature nobody
has built.

In [7]:
t = tags(d)
print(t.to_string(index=False))
print()
print(gaps(t).to_string(index=False))

            tag  mentions  mean_score
           mood        51        3.42
         energy        49        3.67
        harmony        34        3.61
         rhythm        29        3.83
instrumentation        25        4.00
        nothing        19        1.11
         timbre        19        3.44
     production        14        3.64
         vocals         7        3.29

            tag  mentions  mean_score                                                                         families     status
           mood        51        3.42                                                                                — NO FEATURE
         energy        49        3.67                                                                        rmse, zcr      built
        harmony        34        3.61                                    chroma_cens, chroma_cqt, chroma_stft, tonnetz      built
         rhythm        29        3.83                                 tempo, onset_rate, onset_stre

`rhythm` was the second most used word, on pairs scoring 4.12 out of 5 — and FMA
has no rhythm feature at all. Its eleven families are spectral or harmonic.
Tempo, pulse and note density are simply absent, so the model could not tell
whether two tracks share a beat.

`instrumentation` had the same problem: nothing in the feature set says whether a
track is carried by drums or by instruments.

Both were extracted from the mp3s rather than the feature table, because a
feature that is not in the table cannot be recovered from it.

## 5. Two tests, not one

Before the results, the method — because one test is not enough to judge a new
feature, and the difference matters twice below.

**Unsupervised agreement** is what the recommender gets: of a track's ten
nearest neighbours, how many share its genre. This is the existing metric.

**The supervised ceiling** hands a classifier the genre labels and lets it train
on them. Whatever it reaches is roughly the most these columns can say about
genre at all.

Read together:

| ceiling | agreement | reading |
|---|---|---|
| rises | rises | the columns carry new information |
| flat | rises | the same information, rearranged |
| flat | flat | nothing was added |

A feature that lifts agreement while leaving the ceiling flat has not taught the
model anything. It has only made the existing information easier to reach — and
that gain can just as easily come at the cost of something else, as it does
below.

In [8]:
def agreement(F, k=10, exclude_artist=False):
    """Share of the k nearest neighbours sharing the seed's genre."""
    Z = StandardScaler().fit_transform(F)
    n_candidates = 60 if exclude_artist else k + 1
    _, idx = NearestNeighbors(metric="cosine", algorithm="brute").fit(Z).kneighbors(
        Z, n_neighbors=n_candidates)
    scores = []
    for i in range(len(Z)):
        nb = [j for j in idx[i]
              if j != i and (not exclude_artist or artists[j] != artists[i])][:k]
        if nb:
            scores.append((genres[nb] == genres[i]).mean())
    return float(np.mean(scores))


def ceiling(F):
    """What a classifier reaches when handed the labels to train on."""
    clf = make_pipeline(StandardScaler(),
                        RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42))
    return cross_val_score(clf, F, labels, cv=5, n_jobs=-1).mean()


def report(name, F):
    rho, p = human_rho(F)
    print(f"{name:32} {F.shape[1]:4} {agreement(F):9.3f} {ceiling(F):9.3f} "
          f"{rho:9.3f} {p:7.3f}")


print(f"{'space':32} {'cols':>4} {'agree@10':>9} {'ceiling':>9} {'rho':>9} {'p':>7}")
report("timbre", space(TIMBRE))
report("timbre + rhythm", space(TIMBRE, R))
report("mfcc", space(["mfcc"]))
report("mfcc + rhythm", space(["mfcc"], R))

space                            cols  agree@10   ceiling       rho       p


timbre                            224     0.458     0.508       nan     nan


timbre + rhythm                   237     0.472     0.542       nan     nan


mfcc                              140     0.412     0.485       nan     nan


mfcc + rhythm                     153     0.435     0.526       nan     nan


Thirteen rhythm columns lift the ceiling in both spaces — 0.512 to 0.546, and
0.488 to 0.528. The information is new, not rearranged. That is the whole
justification for having gone back to the audio.

### And the same test on the harmonic-percussive split

`modeling/separation.py` splits each track into its harmonic and percussive
halves and writes 81 columns: one energy ratio, and the timbre of each half
separately. The ratio is the measured part of the idea; the two split mfcc
blocks are the hypothesis.

In [9]:
print(f"{'space':32} {'cols':>4} {'agree@10':>9} {'ceiling':>9} {'rho':>9} {'p':>7}")
report("mfcc + rhythm", space(["mfcc"], R))
report("mfcc + rhythm + ratio", space(["mfcc"], pd.concat([R, RATIO], axis=1)))
report("mfcc + rhythm + all 81", space(["mfcc"], pd.concat([R, S], axis=1)))
report("timbre + rhythm", space(TIMBRE, R))
report("timbre + rhythm + ratio", space(TIMBRE, pd.concat([R, RATIO], axis=1)))
report("timbre + rhythm + all 81", space(TIMBRE, pd.concat([R, S], axis=1)))

space                            cols  agree@10   ceiling       rho       p


mfcc + rhythm                     153     0.435     0.526       nan     nan


mfcc + rhythm + ratio             154     0.438     0.533       nan     nan


mfcc + rhythm + all 81            234     0.459     0.534       nan     nan


timbre + rhythm                   237     0.472     0.542       nan     nan


timbre + rhythm + ratio           238     0.474     0.546       nan     nan


timbre + rhythm + all 81          318     0.488     0.546       nan     nan


This is where the second test earns its keep.

The full 81 columns look like the best result in the table — `timbre + rhythm +
all 81` reaches 0.488, higher than anything else here. But the ceiling does not
move: 0.546 to 0.548. The eighty split-mfcc columns are restating what the
unsplit mfcc already said. And the human correlation gets worse, −0.470 to
−0.423, for the same reason the 294 harmony columns hurt: volume dilutes signal.

The single ratio column behaves the other way. It lifts the ceiling, 0.528 to
0.536, and it improves the human correlation to −0.476, the strongest reading in
this notebook.

So one column of the 81 is kept and eighty are discarded. Judged on agreement
alone the decision would have gone the other way.

## Where this leaves the model

Two candidates, both already built. Both keep mfcc, rhythm and the ratio; they
differ only in whether the other 84 timbre columns come along.

In [10]:
from sklearn.decomposition import PCA

A = space(["mfcc"], pd.concat([R, RATIO], axis=1))
B = space(TIMBRE, pd.concat([R, RATIO], axis=1))

built = {}
for name, F in [("A: mfcc + rhythm + ratio", A), ("B: timbre + rhythm + ratio", B)]:
    Z = PCA(n_components=0.90, random_state=42).fit_transform(StandardScaler().fit_transform(F))
    built[name] = (NearestNeighbors(metric="cosine").fit(Z), Z)
    print(f"{name:28} {F.shape[1]:4} columns -> {Z.shape[1]:3} components")

# If the two spaces recommended the same tracks there would be nothing for a
# second listening round to compare, and it would not be worth anyone's evening.
rng = np.random.default_rng(42)
seeds = rng.choice(len(common), 500, replace=False)
(a_index, a_Z), (b_index, b_Z) = built.values()
_, ai = a_index.kneighbors(a_Z[seeds], n_neighbors=11)
_, bi = b_index.kneighbors(b_Z[seeds], n_neighbors=11)
shared = np.array([len(set(a[1:]) & set(b[1:])) for a, b in zip(ai, bi)])
print(f"\nacross 500 seeds the two spaces share {shared.mean():.1f} of their top 10")

A: mfcc + rhythm + ratio      154 columns ->  57 components
B: timbre + rhythm + ratio    238 columns ->  81 components

across 500 seeds the two spaces share 4.5 of their top 10


Half the recommendations differ, so there is a real question to put to listeners.

The genre metrics prefer B. The human scores prefer A, by a wide margin, on 42
ratings from two people. Those two rulers disagree, and this notebook cannot
settle it — the difference between −0.476 and −0.225 is measured on the same
pairs, and separating two dependent correlations needs far more data than
testing either against zero.

### What settles it

A second round, built as a forced choice rather than a rating. One seed, one
recommendation from A and one from B, order randomised, and a single question:
which sounds more related? A within-listener comparison cancels the personal
scale that makes 1-5 ratings noisy, needs on the order of a hundred judgements
rather than a thousand, and asks the question directly instead of inferring it.

Its seeds must be new, and the threshold for calling it a failure has to be
written down before it runs. Otherwise the second round is the first round again
under another name.

### Still outstanding

- four raters, which takes this from 42 ratings to 125 and from 33 pairs to 80
- `mood`, the most used word in the vocabulary, with no feature behind it — and
  probably no feature possible, since it reads as a combination of energy,
  harmony and tempo rather than a thing of its own
- same-artist exclusion, which the other baseline applies and this one does not:
  it accounts for the entire gap between the two, and adding it would move every
  number in the README by about 0.07